## Convert NDPI WSI or TMA images to TIF format with OpenSlide library

In [ ]:
import os
import argparse

import openslide
from PIL import Image
from pathlib import Path
import tifffile as tiff

In [ ]:
def ndpi_to_tiff_full(ndpi_path, tiff_path, level=0):
    """
    将指定 .ndpi 文件转换为一个完整 .tif 文件。
    
    参数：
      - ndpi_path : 输入 NDPI 文件路径（str 或 Path）
      - tiff_path : 输出 TIFF 文件路径（str 或 Path）
      - level     : 金字塔级别，0 = 最高分辨率
    """
    slide = openslide.OpenSlide(str(ndpi_path))
    dims  = slide.level_dimensions[level]
    print(f"Reading level {level} of size {dims[0]} × {dims[1]}")
    region = slide.read_region((0,0), level, dims)
    region = region.convert("RGB")  # 如果你希望灰度或保留通道可修改
    region.save(str(tiff_path), format="TIFF", compression="none")
    slide.close()


def crop_ndpi_region_to_tiff(ndpi_path, output_tif, level, x, y, w, h):
    """
    从 .ndpi 文件的 level 层中裁剪出一个矩形区域，并保存为 TIFF。
    
    参数：
      - ndpi_path   : 输入 .ndpi 文件路径（字符串或 Path）
      - output_tif : 输出 TIFF 文件路径
      - level      : 金字塔级别数（0 为最高分辨率）
      - x, y       : 裁剪区域在 level-0 坐标系中的起点（左上角像素）
      - w, h       : 裁剪区域宽度、高度（像素数，在 level 对应尺度下）
    """
    slide = openslide.OpenSlide(str(ndpi_path))
    # size of full level
    dims = slide.level_dimensions[level]
    print(f"Slide level {level} dimensions: {dims}")
    
    # 检查所选区域是否超出
    if x + w > dims[0] or y + h > dims[1]:
        raise ValueError("Crop region extends beyond the image boundaries!")
    
    # 裁剪区域：read_region 参数 (location, level, size)
    region = slide.read_region((x, y), level, (w, h))
    # region 是 PIL RGBA 图像（带透明通道）
    region = region.convert("RGB")
    # 保存为 TIFF
    region.save(str(output_tif), format="TIFF", compression="none")
    slide.close()
    print(f"Saved cropped region to {output_tif}")

### Transform the full image to TIF

In [ ]:
input_ndpi_dir = "D:/projects/datasets/in-house_ov_data/20250627_WSI_TMA/slide_HE_WSI/ndpi_format"
output_tiff_dir = "D:/projects/datasets/in-house_ov_data/20250627_WSI_TMA/slide_HE_WSI/tif_format"
if not os.path.exists(output_tiff_dir):
    os.makedirs(output_tiff_dir)

level = 0 # 0 for highest resolution

input_ndpi_list = [f for f in os.listdir(input_ndpi_dir) if f.endswith(".ndpi")]
input_ndpi_list.sort()

for input_ndpi_path in input_ndpi_list:
    print(f"Processing {input_ndpi_path}")
    input_ndpi_path = os.path.join(input_ndpi_dir, input_ndpi_path)
    output_tiff_path = os.path.join(output_tiff_dir, input_ndpi_path.replace(".ndpi", ".tif"))
    
    # Transform the full image to TIF
    ndpi_to_tiff_full(input_ndpi_path, output_tiff_path, level=level)

    print(f"Finished processing {input_ndpi_path}")